> **Role:** Step 4 - retrieval + reranking evaluation benchmark
> **Runtime:** Kaggle (GPU); reads the data artifacts from a Kaggle input dataset
> **Writes:** checkpoints and summary under `evaluation_outputs/`
> **Env:** `GEMINI_API_KEY`

# Ibn Baz Fatwa Q&A — Kaggle Evaluation Benchmark

## Four evaluation parts
1. **Test 1:** Original questions → exact fatwa_id metrics (no Gemini).
2. **Test 2:** Original questions → Gemini semantic judge on reranked Top-5 (~500 calls).
3. **Test 3A:** Gemini rewrites each question → exact-ID metrics (~500 rewrite calls).
4. **Test 3B:** Rewritten questions → Gemini semantic judge (~500 calls).

Total Gemini budget: approximately **1,500 requests**. Every Gemini stage checkpoints and can resume.

> The same three data files are published on the Hugging Face dataset [`ZenoAbderrahim/binbaz_fatwas_rag`](https://huggingface.co/datasets/ZenoAbderrahim/binbaz_fatwas_rag). To run outside Kaggle, download them with the optional cell below and point the path variables at `data/`.

In [ ]:
# Optional: fetch the artifacts from Hugging Face instead of a Kaggle input dataset.
# !pip install -q huggingface_hub
# from huggingface_hub import hf_hub_download
# for name in ("binbaz_clean.csv", "qwen3_06b_title_answer_embeddings.pt", "embedding_metadata.json"):
#     hf_hub_download("ZenoAbderrahim/binbaz_fatwas_rag", name, repo_type="dataset", local_dir="data")

In [6]:
import os, json, time, re, ast, gc, shutil
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/abderrahimzineddine/binbaz-fatwa-artifacts")  
CLEAN_CSV = DATA_DIR / "binbaz_clean.csv"
EMBEDDINGS_PT = DATA_DIR / "qwen3_06b_title_answer_embeddings.pt"
METADATA_JSON = DATA_DIR / "embedding_metadata.json"

EMBEDDER_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_MODEL_NAME = "ALJIACHI/Mizan-Rerank-V2"
GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

N_EVAL, SEED = 30, 42
TOP_K, RERANK_TOP_N = 20, 5
MAX_SEQ_LENGTH = 1024
EMBED_QUERY_BATCH_SIZE = 32
RERANKER_BATCH_SIZE = 32

OUT_DIR = Path("/kaggle/working/evaluation_outputs_30"); OUT_DIR.mkdir(exist_ok=True)
EVAL_SAMPLE_CSV = OUT_DIR / "evaluation_sample.csv"
T1 = OUT_DIR / "test1_original_exact.csv"
T2 = OUT_DIR / "test2_original_semantic.csv"
REWRITES = OUT_DIR / "rewritten_queries.csv"
T3A = OUT_DIR / "test3a_rewritten_exact.csv"
T3B = OUT_DIR / "test3b_rewritten_semantic.csv"
SUMMARY = OUT_DIR / "evaluation_summary.json"

for p in [CLEAN_CSV, EMBEDDINGS_PT, METADATA_JSON]: assert p.exists(), p
print("GPUs:", __import__("torch").cuda.device_count())

GPUs: 2


In [4]:
!pip install -q -U sentence-transformers google-genai
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types
print(torch.cuda.get_device_name(0))
if torch.cuda.device_count() > 1: print(torch.cuda.get_device_name(1))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 13.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 12.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incomp

In [7]:
df = pd.read_csv(CLEAN_CSV)

# CSV round-tripping turns empty cells into NaN floats. Normalize text fields once.
TEXT_COLS = ["title", "question", "answer", "categories", "links"]
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

df["title_answer_text"] = df["title"] + " — " + df["answer"]

metadata = json.load(open(METADATA_JSON, encoding="utf-8"))
embeddings = torch.load(EMBEDDINGS_PT, map_location="cpu", weights_only=True).float()
assert len(df) == embeddings.shape[0] == metadata["n_rows"]
assert (df["fatwa_id"].astype(int).to_numpy() == np.arange(len(df))).all(), "CSV order/fatwa_id alignment changed"
embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
print(len(df), embeddings.shape, metadata)


19727 torch.Size([19727, 1024]) {'model': 'Qwen/Qwen3-Embedding-0.6B', 'embed_dim': 1024, 'max_seq_length': 1024, 'n_rows': 19727, 'n_truncated': 864, 'source': 'Ibn Baz', 'generated_at_utc': '2026-09-08T13:26:56.191210+00:00'}


In [8]:
if EVAL_SAMPLE_CSV.exists():
    eval_df = pd.read_csv(EVAL_SAMPLE_CSV)
else:
    eval_df = df[df.question.fillna("").str.strip().ne("")].sample(N_EVAL, random_state=SEED).reset_index(drop=True)
    eval_df["eval_id"] = range(len(eval_df))
    eval_df.to_csv(EVAL_SAMPLE_CSV, index=False)
display(eval_df[["eval_id","fatwa_id","title","question"]].head())

,eval_id,fatwa_id,title,question
0,0,18380,حكم الزواج بامرأة رضعت من أم الزوج رضعتين,تقول في قضية أخرى: فتاة رضعت من امرأةٍ وهي صغي...
1,1,16688,كيف يجمع المؤمن بين الخوف والرجاء؟,يقول: قرأت في أحد الكتب، وهو كتاب حكم وإرشادات...
2,2,14809,حكم استعمال حبوب منع الحمل لتنظيم الأسرة,نعود مع مطلع هذه الحلقة إلى رسالة المستمع: (ع....
3,3,218,"معنى ""إذا قامت الساعة وفي يد أحدكم فسيلة""",ما درجة هذا الحديث: إذا قامت الساعةُ وفي يد أح...
4,4,11631,حكم تأخير صلاة الفجر إلى بعد الشروق بسبب الإرهاق,يقول: إنني أنام أحيانًا عن صلاة الفجر، وهذا ال...


In [9]:
embedder = SentenceTransformer(EMBEDDER_MODEL_NAME, device="cpu")
embedder.max_seq_length = MAX_SEQ_LENGTH

def encode_queries(texts):
    texts = list(texts)
    if torch.cuda.device_count() >= 2:
        pool = embedder.start_multi_process_pool(target_devices=["cuda:0", "cuda:1"])
        try:
            # encode_multi_process is deprecated; encode() now accepts the pool directly.
            x = embedder.encode(
                texts,
                pool=pool,
                batch_size=EMBED_QUERY_BATCH_SIZE,
                show_progress_bar=True,
            )
        finally:
            embedder.stop_multi_process_pool(pool)
    else:
        x = embedder.encode(texts, batch_size=EMBED_QUERY_BATCH_SIZE, show_progress_bar=True)
    return torch.from_numpy(np.asarray(x)).float()

def retrieve(queries):
    q = torch.nn.functional.normalize(encode_queries(queries), p=2, dim=1)
    scores = q @ embeddings.T
    return torch.topk(scores, k=TOP_K, dim=1)

print("Embedder ready")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedder ready


In [10]:
devices = [f"cuda:{i}" for i in range(min(2, torch.cuda.device_count()))]
assert devices, "No CUDA GPU available for reranking"

rerankers = {}
for device in devices:
    print("Loading", device)
    rerankers[device] = CrossEncoder(
        RERANKER_MODEL_NAME,
        device=device,
        trust_remote_code=True,
        max_length=MAX_SEQ_LENGTH,
    )

def rerank(query, indices, retrieval_scores, model):
    query = str(query)
    indices = np.asarray(indices, dtype=int)

    pairs = [[query, df.iloc[int(i)]["title_answer_text"]] for i in indices]
    scores = np.asarray(
        model.predict(
            pairs,
            batch_size=RERANKER_BATCH_SIZE,
            show_progress_bar=False,
        )
    )

    order = np.argsort(-scores)
    return (
        df.iloc[indices]["fatwa_id"].astype(int).tolist(),
        [float(x) for x in retrieval_scores],
        df.iloc[indices[order]]["fatwa_id"].astype(int).tolist(),
        [float(scores[i]) for i in order],
    )

def run_pipeline(queries, expected_ids, out_path):
    queries = list(queries)
    expected_ids = list(expected_ids)

    if out_path.exists():
        old = pd.read_csv(out_path)
        done = set(old.eval_id.astype(int)) if len(old) else set()
    else:
        old = pd.DataFrame()
        done = set()

    rows = []
    vals, inds = retrieve(queries)
    vals, inds = vals.numpy(), inds.numpy()

    for i, query in enumerate(tqdm(queries, desc="Reranking")):
        eid = int(eval_df.iloc[i].eval_id)
        if eid in done:
            continue

        rid, rs, rrid, rrs = rerank(
            query,
            inds[i],
            vals[i],
            rerankers[devices[i % len(devices)]],
        )

        rows.append(dict(
            eval_id=eid,
            query=str(query),
            expected_fatwa_id=int(expected_ids[i]),
            retrieved_ids=rid,
            retrieval_scores=rs,
            reranked_ids=rrid,
            reranker_scores=rrs,
        ))

        if len(rows) % 10 == 0:
            old = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)
            old.to_csv(out_path, index=False, encoding="utf-8")
            rows = []

    if rows:
        old = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)

    old = old.drop_duplicates(subset=["eval_id"], keep="last").sort_values("eval_id").reset_index(drop=True)
    old.to_csv(out_path, index=False, encoding="utf-8")
    return old


Loading cuda:0


modules.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Loading cuda:1


Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

In [11]:
# TEST 1 — ORIGINAL QUESTION / EXACT ID
test1_df = run_pipeline(eval_df.question.fillna(""), eval_df.fatwa_id.astype(int), T1)
print("Saved", T1)

def L(x): return x if isinstance(x,list) else ast.literal_eval(x)
def exact_metrics(x):
    exp=x.expected_fatwa_id.astype(int).tolist(); ret=x.retrieved_ids.apply(L); rer=x.reranked_ids.apply(L)
    def rec(lists,k): return float(np.mean([e in list(map(int,a[:k])) for e,a in zip(exp,lists)]))
    def mrr(lists):
        z=[]
        for e,a in zip(exp,lists):
            a=list(map(int,a)); z.append(1/(a.index(e)+1) if e in a else 0)
        return float(np.mean(z))
    return {"embedder_R@1":rec(ret,1),"embedder_R@5":rec(ret,5),"embedder_R@10":rec(ret,10),"embedder_R@20":rec(ret,20),"embedder_MRR":mrr(ret),"reranker_R@1":rec(rer,1),"reranker_R@3":rec(rer,3),"reranker_R@5":rec(rer,5),"reranker_MRR":mrr(rer)}
test1_metrics=exact_metrics(test1_df); display(pd.DataFrame([test1_metrics]).T)

Chunks:   0%|          | 0/15 [00:00<?, ?it/s]

Reranking:   0%|          | 0/30 [00:00<?, ?it/s]

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Saved /kaggle/working/evaluation_outputs_30/test1_original_exact.csv


,0
embedder_R@1,0.333333
embedder_R@5,0.533333
embedder_R@10,0.566667
embedder_R@20,0.666667
embedder_MRR,0.411263
reranker_R@1,0.366667
reranker_R@3,0.500000
reranker_R@5,0.566667
reranker_MRR,0.442910


In [12]:
# GEMINI HELPERS
assert GEMINI_API_KEY, "Add GEMINI_API_KEY as a Kaggle secret/environment variable"
client=genai.Client(api_key=GEMINI_API_KEY)

def gcall(prompt):
    delay=2
    for attempt in range(7):
        try:
            r=client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=types.GenerateContentConfig(temperature=0.0,max_output_tokens=1024))
            return r.text.strip()
        except Exception as e:
            if attempt==6: raise
            print("Retry",e); time.sleep(delay); delay=min(delay*2,60)

def jparse(s):
    s=re.sub(r"^```(?:json)?\s*|\s*```$","",s.strip())
    return json.loads(s)

In [ ]:
print("GEMINI_API_KEY:", "set" if GEMINI_API_KEY else "NOT set - export GEMINI_API_KEY")

In [14]:
# TEST 2 — ORIGINAL QUESTION / SEMANTIC JUDGE
# Deliberately use marker replacement instead of str.format(), because the required JSON
# example contains literal braces that str.format() interprets as placeholders.
JUDGE = """You evaluate a fatwa retrieval system.

User question:
<<QUERY>>

For each candidate, classify exactly as one of:
- answers-it: directly answers the user's question or provides the ruling needed.
- related-only: discusses a related topic but does not adequately answer the question.
- unrelated: does not meaningfully address the question.

Evaluate each candidate independently. Return STRICT JSON only, with exactly this schema:
{
  "results": [
    {"rank": 1, "verdict": "answers-it", "reason": "brief reason"}
  ]
}

Candidates:
<<CANDIDATES>>
"""

def L(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        return ast.literal_eval(x)
    return []

def candidates(ids):
    out = []
    for rank, fid in enumerate(L(ids)[:RERANK_TOP_N], 1):
        r = df.loc[df.fatwa_id == int(fid)].iloc[0]
        out.append(
            f"--- Candidate rank {rank} | fatwa_id={int(fid)} ---\n"
            f"Title: {r['title']}\n"
            f"Question: {r['question']}\n"
            f"Answer: {r['answer']}"
        )
    return "\n\n".join(out)

def make_judge_prompt(query, ids):
    return (
        JUDGE
        .replace("<<QUERY>>", str(query))
        .replace("<<CANDIDATES>>", candidates(ids))
    )

def judge(results, out_path):
    if out_path.exists():
        old = pd.read_csv(out_path)
        done = set(old.eval_id.astype(int)) if len(old) else set()
    else:
        old = pd.DataFrame()
        done = set()

    rows = []

    for _, x in tqdm(results.iterrows(), total=len(results), desc="Gemini judge"):
        eval_id = int(x.eval_id)
        if eval_id in done:
            continue

        prompt = make_judge_prompt(x.query, x.reranked_ids)
        raw = gcall(prompt)

        try:
            parsed = jparse(raw)
            verdict = parsed.get("results", [])
            if not isinstance(verdict, list):
                raise ValueError("'results' is not a list")
        except Exception as e:
            print(f"\nParse failed for eval_id={eval_id}: {e}")
            verdict = []

        rows.append(dict(
            eval_id=eval_id,
            query=str(x.query),
            expected_fatwa_id=int(x.expected_fatwa_id),
            reranked_ids=L(x.reranked_ids)[:RERANK_TOP_N],
            judge_results=json.dumps(verdict, ensure_ascii=False),
            raw_response=raw,
        ))

        if len(rows) % 10 == 0:
            old = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)
            old.to_csv(out_path, index=False, encoding="utf-8")
            rows = []

    if rows:
        old = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)

    old = old.drop_duplicates(subset=["eval_id"], keep="last").sort_values("eval_id").reset_index(drop=True)
    old.to_csv(out_path, index=False, encoding="utf-8")
    return old

def semantic_metrics(x):
    out = {}
    for k in [1, 3, 5]:
        ok = []
        for _, r in x.iterrows():
            vs = L(r.judge_results)
            d = {int(v.get("rank", -1)): v.get("verdict") for v in vs if isinstance(v, dict)}
            ok.append(any(d.get(i) == "answers-it" for i in range(1, k + 1)))
        out[f"semantic_success@{k}"] = float(np.mean(ok))
    return out

# Smoke test one request before spending the full budget:
# sample_prompt = make_judge_prompt(test1_df.iloc[0].query, test1_df.iloc[0].reranked_ids)
# print(sample_prompt)
# print(jparse(gcall(sample_prompt)))

test2_df = judge(test1_df, T2)
test2_metrics = semantic_metrics(test2_df)
display(pd.DataFrame([test2_metrics]).T)


Gemini judge:   0%|          | 0/30 [00:00<?, ?it/s]

Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABL

,0
semantic_success@1,0.666667
semantic_success@3,0.833333
semantic_success@5,0.900000


In [15]:
# GENERATE 500 GEMINI REWRITES
REWRITE='Rewrite this Islamic question as a realistic Arabic user query. Preserve exact meaning, use different wording, do not answer it, do not add facts. Return ONLY the rewritten question.\n\nOriginal: {q}'

if REWRITES.exists(): rewrites_df=pd.read_csv(REWRITES); done=set(rewrites_df.eval_id)
else: rewrites_df=pd.DataFrame(); done=set()
rows=[]
for _,x in tqdm(eval_df.iterrows(),total=len(eval_df),desc="Gemini rewriting"):
    if int(x.eval_id) in done: continue
    rows.append(dict(eval_id=x.eval_id,fatwa_id=x.fatwa_id,original_question=x.question,rewritten_question=gcall(REWRITE.format(q=x.question))))
    if len(rows)%10==0: rewrites_df=pd.concat([rewrites_df,pd.DataFrame(rows)],ignore_index=True);rewrites_df.to_csv(REWRITES,index=False);rows=[]
if rows: rewrites_df=pd.concat([rewrites_df,pd.DataFrame(rows)],ignore_index=True)
rewrites_df.to_csv(REWRITES,index=False); rewrites_df=rewrites_df.sort_values("eval_id").reset_index(drop=True)
display(rewrites_df.head())

Gemini rewriting:   0%|          | 0/30 [00:00<?, ?it/s]

Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABL

,eval_id,fatwa_id,original_question,rewritten_question
0,0,18380,تقول في قضية أخرى: فتاة رضعت من امرأةٍ وهي صغي...,ما حكم زواج شاب من فتاة رضعت من أمه مرتين، علم...
1,1,16688,يقول: قرأت في أحد الكتب، وهو كتاب حكم وإرشادات...,سماحة الشيخ، اطلعت على عبارة في أحد كتب التوجي...
2,2,14809,نعود مع مطلع هذه الحلقة إلى رسالة المستمع: (ع....,أنا مقيم في إحدى القرى النائية ولدي أطفال صغار...
3,3,218,ما درجة هذا الحديث: إذا قامت الساعةُ وفي يد أح...,ما صحة حديث إذا قامت الساعة وفي يد أحدكم فسيلة؟
4,4,11631,يقول: إنني أنام أحيانًا عن صلاة الفجر، وهذا ال...,سائل يطرح مشكلته ويقول: أحياناً يغلبني النوم ف...


In [16]:
# TEST 3A — REWRITTEN QUESTION / EXACT ID
test3a_df=run_pipeline(rewrites_df.rewritten_question, rewrites_df.fatwa_id.astype(int), T3A)
test3a_metrics=exact_metrics(test3a_df)
display(pd.DataFrame([test3a_metrics]).T)

Chunks:   0%|          | 0/15 [00:00<?, ?it/s]

Reranking:   0%|          | 0/30 [00:00<?, ?it/s]

,0
embedder_R@1,0.266667
embedder_R@5,0.500000
embedder_R@10,0.533333
embedder_R@20,0.600000
embedder_MRR,0.366604
reranker_R@1,0.300000
reranker_R@3,0.433333
reranker_R@5,0.466667
reranker_MRR,0.387963


In [17]:
# TEST 3B — REWRITTEN QUESTION / SEMANTIC JUDGE
test3b_df=judge(test3a_df,T3B)
test3b_metrics=semantic_metrics(test3b_df)
display(pd.DataFrame([test3b_metrics]).T)

Gemini judge:   0%|          | 0/30 [00:00<?, ?it/s]

Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retry 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABL

,0
semantic_success@1,0.600000
semantic_success@3,0.733333
semantic_success@5,0.733333


In [18]:
# FINAL RESULTS + DOWNLOAD PACKAGE
summary={"config":{"n_eval":N_EVAL,"seed":SEED,"top_k":TOP_K,"rerank_top_n":RERANK_TOP_N,"embedder":EMBEDDER_MODEL_NAME,"reranker":RERANKER_MODEL_NAME,"gemini":GEMINI_MODEL},"test1_original_exact":test1_metrics,"test2_original_semantic":test2_metrics,"test3a_rewritten_exact":test3a_metrics,"test3b_rewritten_semantic":test3b_metrics}
json.dump(summary,open(SUMMARY,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
comparison=pd.DataFrame([{"Test":"1 Original exact",**test1_metrics},{"Test":"2 Original semantic",**test2_metrics},{"Test":"3A Rewritten exact",**test3a_metrics},{"Test":"3B Rewritten semantic",**test3b_metrics}])
display(comparison)
archive=shutil.make_archive("/kaggle/working/binbaz_evaluation_results","zip",OUT_DIR)
print("DOWNLOAD:",archive)
print("Files:",[p.name for p in OUT_DIR.iterdir()])

,Test,embedder_R@1,embedder_R@5,embedder_R@10,embedder_R@20,embedder_MRR,reranker_R@1,reranker_R@3,reranker_R@5,reranker_MRR,semantic_success@1,semantic_success@3,semantic_success@5
0,1 Original exact,0.333333,0.533333,0.566667,0.666667,0.411263,0.366667,0.500000,0.566667,0.442910,NaN,NaN,NaN
1,2 Original semantic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.666667,0.833333,0.900000
2,3A Rewritten exact,0.266667,0.500000,0.533333,0.600000,0.366604,0.300000,0.433333,0.466667,0.387963,NaN,NaN,NaN
3,3B Rewritten semantic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.600000,0.733333,0.733333


DOWNLOAD: /kaggle/working/binbaz_evaluation_results.zip
Files: ['test3a_rewritten_exact.csv', 'evaluation_sample.csv', 'test1_original_exact.csv', 'evaluation_summary.json', 'test2_original_semantic.csv', 'test3b_rewritten_semantic.csv', 'rewritten_queries.csv']
